# Preparation: Baseline vs. Pipeline A

Builds `baseline.csv` and `pipelineA.csv` and merges both onto the slimmed Gold-Standard (`../gs_slim.json`).

In [3]:
import pandas as pd
import numpy as np

from pathlib import Path

In [4]:
%run ../baseline/flattening.py
%run ../PipelineA/flattening.py
print("="*120)
print("flattening done.")
print("="*120)


  Flattening 54 JSONs => /Users/jannikkuhlmann/VSC/LaTeX/2026_BA_Code/evaluations/baseline/baseline.csv
  Reports    : 54
  Zeilen     : 603
  Fehler     : 0
  Gespeichert: /Users/jannikkuhlmann/VSC/LaTeX/2026_BA_Code/evaluations/baseline/baseline.csv

  Flattening 54 JSONs => /Users/jannikkuhlmann/VSC/LaTeX/2026_BA_Code/evaluations/PipelineA/PipelineA-Answers.csv
  Reports    : 53
  Zeilen     : 647
  Fehler     : 0
  Gespeichert: /Users/jannikkuhlmann/VSC/LaTeX/2026_BA_Code/evaluations/PipelineA/PipelineA-Answers.csv

Total Errors: 0
[OK] No errors detected.
flattening done.


## Import slimmed down Gold-Standard

In [5]:
gs = pd.read_json("../gs_slim.json")
gs["year"] = gs["year"].astype(str) #Correction needed for e.g. fiscal years "FY 2021/2022"

## Import the Extractions

In [6]:
baseline_df  = pd.read_csv(Path("./../baseline/baseline.csv"))
pipelineA_df = pd.read_csv(Path("./../PipelineA/PipelineA-Answers.csv"))

df_to_merge = [
    (baseline_df, "_Base"),
    (pipelineA_df, "_A"),
]

In [5]:
# Year normalization RegEx-Script

import re

def normalize_year(raw: str, years_present: set[int] | None = None) -> tuple[int | None, str]:
    """Map a raw fiscal-year label to a calendar year. Returns (year, rule_applied)."""

    if isinstance(raw, float):
        if pd.isna(raw):
            return None, "unparseable"
        raw = int(raw) if raw.is_integer() else raw

    label = str(raw).strip().upper().removeprefix("FY").strip()

    if re.fullmatch(r"\d{4}", label):
        return int(label), "plain"
    if re.fullmatch(r"\d{2}", label):
        return 2000 + int(label), "fy_2digit"

    m = re.fullmatch(r"(\d{4})/(\d{1,4})", label)
    if m:
        left, right = m.groups()
        if len(right) == 4:
            return int(left), "range_former_year"  # 2021/2022 -> 2021
        candidates = {int(left), int(left) + 1}
        if years_present:
            hit = candidates & years_present
            if len(hit) == 1:
                return hit.pop(), "fy_end_month_context"
        return int(left), "fy_end_month_fallback"

    return None, "unparseable"

In [6]:
# Get years from extractions
years_in_extractions = set()
for df, _ in df_to_merge:
    years_in_extractions.update(df["year"].unique().tolist())

# Create ynorm DataFrames to preserve the original ones
baseline_ynorm  = baseline_df.copy()
pipelineA_ynorm = pipelineA_df.copy()

df_to_merge_ynorm = [
    (baseline_ynorm, "_Base"),
    (pipelineA_ynorm, "_A"),
]

# Now normalization via the RegEx Script from above
for df, _ in df_to_merge_ynorm:
    df["year"], df["year_rule"] = zip(*df["year"].apply(
        normalize_year,
        years_present=years_in_extractions
    ))

## Sanity check: report names

In [7]:
all_extracted = set()
for df, _ in df_to_merge:
    all_extracted.update(df["report_name"].unique())

in_ext_not_gs = sorted(all_extracted - set(gs["report_name"]))
in_gs_not_ext = sorted(set(gs["report_name"]) - all_extracted)

print(f"In extractions, nicht in GS: {len(in_ext_not_gs)} ==> {"OK" if len(in_ext_not_gs) == 0 else "NOK"}")
for r in in_ext_not_gs: print(" ", r)

print(f"\nIn GS, nicht in extractions: {len(in_gs_not_ext)} ==> {"OK" if len(in_gs_not_ext) == 0 else "NOK"}")
for r in in_gs_not_ext: print(" ", r)

In extractions, nicht in GS: 0 ==> OK

In GS, nicht in extractions: 0 ==> OK


## Matching Extractions to Gold-Standard Scheme

### Before ynorm

In [8]:
years = set()

for df, _ in df_to_merge:
    years.update(df["year"].unique().tolist()) # update() = add() but for all elements

print(sorted(years))

['2012', '2014', '2015', '2016', '2017', '2017_pro_rata', '2018', '2019', '2019/2020', '2020', '2020/2021', '2021', '2021/2022', '2022', 'FY 2021', 'FY 2022', 'FY16', 'FY17', 'FY18', 'FY19', 'FY20', 'FY2017', 'FY2018', 'FY2019', 'FY2020', 'FY2020/3', 'FY2021', 'FY2022']


### After ynorm

In [9]:
years = set()

for df, _ in df_to_merge_ynorm:
    years.update(df["year"].unique().tolist()) # update() = add() but for all elements

print(sorted(years, key=lambda y: (y is None, y)))

[2016, 2017, 2018, 2019, 2020, 2021, 2022, nan, 2012, 2014, 2015]


## Mapping special occurences

In [10]:
scope_mapping = {
    "scope_1": "1",
    "scope_2_location_based": "2lb",
    "scope_2_market_based" : "2mb",
    "scope_3" : "3"
}

# Prints out only if sth. goes wrong
for df, sf in df_to_merge:
    # Replace scope definitons to match Gold-Standard
    df['scope'] = df['scope'].replace(scope_mapping)

    # Ensure every value column is a float64 to match Gold-Standard
    converted = pd.to_numeric(
        df['value'].astype(str).str.replace(",", "", regex=False),
        errors="coerce"
    )

# Alle Werte, die nicht konvertierbar sind (inkl. urspruengliche NaNs)
all_failed = df['value'][converted.isna()]
print(f"Total NaN nach Konversion: {all_failed.notna().sum()}")

Total NaN nach Konversion: 0


In [11]:


# Prints out only if sth. goes wrong
for df, sf in df_to_merge_ynorm:
    # Replace scope definitons to match Gold-Standard
    df['scope'] = df['scope'].replace(scope_mapping)

    # Ensure every value column is a float64 to match Gold-Standard
    converted = pd.to_numeric(
        df['value'].astype(str).str.replace(",", "", regex=False),
        errors="coerce"
    )

# Alle Werte, die nicht konvertierbar sind (inkl. urspruengliche NaNs)
all_failed = df['value'][converted.isna()]
print(f"Total NaN nach Konversion: {all_failed.notna().sum()}")

Total NaN nach Konversion: 0


## Merging Extraction Values and Gold-Standard

In [12]:
merge_on = ["report_name", "scope", "year"]
agg_cols = ["value", "unit", "label"]

In [13]:
merged = gs.copy() # Setting a starting point for the loop. Everything has to be mapped to the Gold-Standard

for df, sf in df_to_merge:
    agg = (
        df.groupby(merge_on)[agg_cols]
        .agg(list)
        .reset_index()
        .rename(columns={col: f"{col}{sf}" for col in agg_cols})
    )
    merged = pd.merge(merged, agg, on=merge_on, how="left")

merged.info()

<class 'pandas.DataFrame'>
RangeIndex: 2208 entries, 0 to 2207
Data columns (total 17 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   report_name      2208 non-null   str    
 1   year             2208 non-null   str    
 2   scope            2208 non-null   str    
 3   page             490 non-null    float64
 4   value            489 non-null    float64
 5   unit             488 non-null    str    
 6   unit_normalized  488 non-null    str    
 7   label            489 non-null    str    
 8   status           2208 non-null   str    
 9   scopes_present   2208 non-null   object 
 10  years_present    2208 non-null   object 
 11  value_Base       456 non-null    object 
 12  unit_Base        456 non-null    object 
 13  label_Base       456 non-null    object 
 14  value_A          452 non-null    object 
 15  unit_A           452 non-null    object 
 16  label_A          452 non-null    object 
dtypes: float64(2), object(8),

In [14]:
merged_ynorm = gs.copy()
merged_ynorm["year"] = merged_ynorm["year"].astype(int)

for df, sf in df_to_merge_ynorm:
    agg = (
        df.groupby(merge_on)[agg_cols]
        .agg(list)
        .reset_index()
        .rename(columns={col: f"{col}{sf}" for col in agg_cols})
    )
    merged_ynorm = pd.merge(merged_ynorm, agg, on=merge_on, how="left")

merged_ynorm.info()

<class 'pandas.DataFrame'>
RangeIndex: 2208 entries, 0 to 2207
Data columns (total 17 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   report_name      2208 non-null   str    
 1   year             2208 non-null   int64  
 2   scope            2208 non-null   str    
 3   page             490 non-null    float64
 4   value            489 non-null    float64
 5   unit             488 non-null    str    
 6   unit_normalized  488 non-null    str    
 7   label            489 non-null    str    
 8   status           2208 non-null   str    
 9   scopes_present   2208 non-null   object 
 10  years_present    2208 non-null   object 
 11  value_Base       527 non-null    object 
 12  unit_Base        527 non-null    object 
 13  label_Base       527 non-null    object 
 14  value_A          526 non-null    object 
 15  unit_A           526 non-null    object 
 16  label_A          526 non-null    object 
dtypes: float64(2), int64(1), 

#### Rows where no value is present must be from Type "NaN" for better analysis; not "None" because the object is missing.

In [15]:
pipeline_cols = (
    [f"value{s}" for _, s in df_to_merge] +
    [f"unit{s}"  for _, s in df_to_merge] +
    [f"label{s}" for _, s in df_to_merge]
)

for col in pipeline_cols:
    merged[col] = merged[col].apply(
    lambda x: np.nan if x is None else x
)

for col in pipeline_cols:
    merged_ynorm[col] = merged_ynorm[col].apply(
    lambda x: np.nan if x is None else x
)

### Rearranging Columns

In [16]:

gs_cols = [
    'report_name', 'status', 'scopes_present', 'years_present', # Per-Report granularity
    'year', 'scope',                                            # Per category
    'page', 'value', 'unit',                                    # About the value as-is in the report
    'unit_normalized', 'label',                                 # Additional information
]

pipeline_cols = (
    [f"value{s}" for _, s in df_to_merge] +
    [f"unit{s}"  for _, s in df_to_merge] +
    [f"label{s}" for _, s in df_to_merge]
)

merged = merged[gs_cols + pipeline_cols]
merged_ynorm = merged_ynorm[gs_cols + pipeline_cols]

merged.info()

<class 'pandas.DataFrame'>
RangeIndex: 2208 entries, 0 to 2207
Data columns (total 17 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   report_name      2208 non-null   str    
 1   status           2208 non-null   str    
 2   scopes_present   2208 non-null   object 
 3   years_present    2208 non-null   object 
 4   year             2208 non-null   str    
 5   scope            2208 non-null   str    
 6   page             490 non-null    float64
 7   value            489 non-null    float64
 8   unit             488 non-null    str    
 9   unit_normalized  488 non-null    str    
 10  label            489 non-null    str    
 11  value_Base       456 non-null    object 
 12  value_A          452 non-null    object 
 13  unit_Base        456 non-null    object 
 14  unit_A           452 non-null    object 
 15  label_Base       456 non-null    object 
 16  label_A          452 non-null    object 
dtypes: float64(2), object(8),

## Saving created DataFrame

In [ ]:
# merged.to_json("Baseline-PipelineA.json", index=False, orient="records", indent=4)
merged_ynorm.to_json("Baseline-PipelineA_ynorm.json", index=False, orient="records", indent=4)

## To later read back the files:
# pd.read_csv("...csv")   # Lists as Strings. Needs ast.literal_eval
# pd.read_json("...json", orient="records")  # Lists instant usable